# 03 · Reconciling grid forecasts to country totals

**What this teaches:** how `views_frames_reconcile` makes PRIO-GRID (`pgm`) forecasts
sum, **per posterior draw**, to their country (`cm`) totals — `reconcile_proportional`
and the `ReconciliationModule` orchestrator — with conservation, zero-preservation,
joint sampling and the bit-identity provenance. And, honestly: what reconciliation
*guarantees* (coherence) versus what it does **not** (it is a pragmatic approximation,
and bit-identity to the old path proves *faithful relocation*, **not** method quality).

**Audience:** anyone wiring reconciliation into a served pipeline, or deciding whether
proportional top-down is good enough.

> **How to read this notebook.** Public, **frozen** `views_frames` /
> `views_frames_reconcile` API only, on **synthetic** data with a *known* true country
> split (`notebooks/_synthetic.py`), so we can ask whether reconciliation moves estimates
> toward the truth. Fixed seeds; *Run All* < 1 min; static plots; toy lattice only (no
> real geography — ADR-001).

### Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

for _p in (Path.cwd(), Path.cwd() / "notebooks"):
    if (_p / "_synthetic.py").exists():
        sys.path.insert(0, str(_p))
        break
import _synthetic as syn

from views_frames import PredictionFrame, SpatialLevel
from views_frames_reconcile import ReconciliationModule, reconcile_proportional
from views_frames_reconcile.conformance import assert_reconcile_contract
from views_frames_summarize import aggregate_distributions, expected_shortfall, hdi

np.set_printoptions(precision=3, suppress=True)

# one scenario reused throughout: a 4x4 grid lattice, 3 months, 2 countries
sc = syn.cm_pgm_scenario(n_samples=512, seed=0, lattice=(4, 4), n_months=3, n_countries=2)

## 1 · The problem

A grid model and a country model are trained separately, so the grid cells of a country
do **not** sum to the country forecast. Downstream consumers need them *coherent* — a
food-security map that doesn't add up to the national number is not usable.

Reconciliation enforces that. Crucially it works **per posterior draw**, in distribution,
not on point estimates: within each draw, the grid cells of a country are rescaled to sum
to that draw's country total. The result is a new grid posterior that is **conservation-
correct sample-by-sample**, so every downstream summary (HDI, exceedance, tail risk) stays
internally consistent.

## 2 · `reconcile_proportional` — the leaf operation

The numpy core: `reconcile_proportional(grid, country)` rescales `grid` `(S, cells)` so
each draw sums to `country` `(S,)`, **top-down by forecast proportion**. Three guarantees,
shown on a tiny explicit example: per-draw conservation, **zeros stay zero**, and the
**all-zero-draw edge** (a draw where every cell is 0 has no proportions to scale — it stays
all-zero rather than dividing by zero).

In [ ]:
rng = np.random.default_rng(0)
S, C = 4000, 4
grid = rng.gamma(2.0, 1.5, size=(S, C)).astype(np.float32)
grid[:, 3] = 0.0                       # cell 3 is a structural zero (quiet cell)
grid[0, :] = 0.0                       # draw 0 is all-zero (the edge case)
country = rng.gamma(20.0, 1.5, size=S).astype(np.float32)  # country forecast per draw
country[0] = 0.0

recon = reconcile_proportional(grid, country)

draw_sums = recon.sum(axis=1)
print("per-draw conservation (active draws): max |sum - country| =",
      f"{np.abs(draw_sums[1:] - country[1:]).max():.2e}")
print("zero cell stays zero                 :", bool((recon[:, 3] == 0).all()))
print("all-zero draw stays all-zero         :", bool((recon[0] == 0).all()))
print("non-negative                         :", bool((recon >= 0).all()))

## 3 · Make it visible

On the scenario, reconcile the whole grid (via the module, §4) and look at one country in
one month: the per-cell means barely move (proportions are preserved) but the **per-draw
country sum** collapses from the grid model's own spread onto the country forecast — that
collapse *is* reconciliation.

In [ ]:
module = ReconciliationModule(sc.map_keys, sc.map_vals)
recon_pgm = module.reconcile(sc.cm_prediction, sc.pgm_prediction)

# pick country 70, month 500
cm_units = sc.pgm_index.cross_level_align_arrays(sc.map_keys, sc.map_vals, SpatialLevel.CM).unit
rows = np.where((sc.pgm_index.time == 500) & (cm_units == 70))[0]
ci = int(np.where((sc.cm_index.time == 500) & (sc.cm_index.unit == 70))[0][0])

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
x = np.arange(len(rows))
axes[0].bar(x - 0.2, sc.pgm_prediction.values[rows].mean(1), 0.4, label="raw")
axes[0].bar(x + 0.2, recon_pgm.values[rows].mean(1), 0.4, label="reconciled")
axes[0].set_title("per-cell mean (proportions kept)", fontsize=9)
axes[0].set_xlabel("grid cell"); axes[0].legend(fontsize=8)
axes[1].hist(sc.pgm_prediction.values[rows].sum(0), bins=40, alpha=0.6, label="raw grid sum")
axes[1].hist(recon_pgm.values[rows].sum(0), bins=40, alpha=0.6, label="reconciled sum")
axes[1].hist(sc.cm_prediction.values[ci], bins=40, alpha=0.6, label="country forecast")
axes[1].set_title("per-draw country sum", fontsize=9); axes[1].legend(fontsize=8); axes[1].set_yticks([])
fig.suptitle("country 70, month 500 — reconciliation snaps the grid sum onto the country", fontsize=10)
plt.tight_layout(); plt.show()

## 4 · `ReconciliationModule` — the orchestrator

The module holds the **injected** `(time, priogrid) → country` mapping (geography is
injected, never embedded or fetched — ADR-014) and applies it: `reconcile(cm, pgm)`
validates the inputs and returns a **new** pgm frame (de-mutated — the caller's frame is
never touched). It **fails loud** when the inputs are inconsistent.

In [ ]:
before = sc.pgm_prediction.values.copy()
out = module.reconcile(sc.cm_prediction, sc.pgm_prediction)
print("returns a new frame, input de-mutated :", np.array_equal(sc.pgm_prediction.values, before))
print("output stays at PGM level             :", out.index.level is SpatialLevel.PGM)

# fail-loud: a mapping that doesn't cover every grid (time, unit)
bad = ReconciliationModule(sc.map_keys[:2], sc.map_vals[:2])
try:
    bad.reconcile(sc.cm_prediction, sc.pgm_prediction)
except ValueError as e:
    print("fails loud on an incomplete mapping   :", str(e)[:58], "...")

# fail-loud: levels swapped (cm where pgm expected)
try:
    module.reconcile(sc.pgm_prediction, sc.pgm_prediction)
except ValueError as e:
    print("fails loud on a wrong-level input     :", str(e)[:58], "...")

## 5 · Contract properties

The package *publishes* its contract — a consumer re-runs `assert_reconcile_contract` in
its own CI. And the mapping is genuinely load-bearing: change the injected
`(time, unit) → country` map and the reconciliation changes, because each cell is grouped
under (and scaled to) a *different* country total.

In [ ]:
assert_reconcile_contract(sc.cm_prediction, sc.pgm_prediction, sc.map_keys, sc.map_vals)
print("all reconcile-contract laws hold ✓ (sum-to-country, zero-preservation, "
      "non-negativity, level/shape)")

# mapping sensitivity: flip every grid cell to the OTHER country
flipped_vals = np.where(sc.map_vals == sc.countries[0], sc.countries[1], sc.countries[0])
recon_flipped = ReconciliationModule(sc.map_keys, flipped_vals).reconcile(
    sc.cm_prediction, sc.pgm_prediction
)
moved = np.abs(recon_pgm.values - recon_flipped.values).mean()
print(f"\nre-injecting a different mapping changes the result (mean |Δ| = {moved:.3f}) — "
      "the mapping is injected and load-bearing, not decorative.")

## 6 · Joint sampling & subadditivity

Because reconciliation is per-draw, the country distribution is a **joint** sum — so, as
in notebook 02 §8, you must aggregate the *distribution* then summarize. A consequence for
risk: expected-shortfall is **subadditive** — the worst-case of the joint country total is
*less than* the sum of the cells' worst-cases (the cells' bad draws don't all line up). So
`ES(aggregate) ≠ Σ ES(cell)`; summing per-cell tail risk **overstates** country risk.

In [ ]:
# raw grid, country 70, month 500
country_dist = aggregate_distributions(sc.pgm_prediction, sc.mapping, SpatialLevel.CM)
crow = int(np.where((country_dist.index.time == 500) & (country_dist.index.unit == 70))[0][0])

es_joint = expected_shortfall(country_dist, [0.1])[crow, 0]        # ES of the joint sum
es_cells = expected_shortfall(sc.pgm_prediction, [0.1])[rows, 0].sum()  # sum of per-cell ES
print(f"worst-10% expected shortfall, country 70:")
print(f"  ES(aggregate)  (joint, correct) : {es_joint:8.1f}")
print(f"  Σ ES(cell)     (naive sum)      : {es_cells:8.1f}")
print(f"  diversification (Σ - joint)     : {es_cells - es_joint:8.1f}  (subadditivity)")

## 7 · Provenance — bit-identity is *faithful relocation*, not method validation

`reconcile_proportional` is a **byte-faithful numpy port** of views-reporting's torch
reconciler, proven bit-identical to a frozen torch oracle
(`scripts/verify_reconcile_parity.py --oracle` → `0.000e+00` error; the 136-case
head-to-head). The reconciler is also deterministic — same inputs, same bytes out:

In [ ]:
a = module.reconcile(sc.cm_prediction, sc.pgm_prediction).values
b = module.reconcile(sc.cm_prediction, sc.pgm_prediction).values
print("deterministic (bit-identical reruns):", np.array_equal(a, b))

**Read this carefully.** Bit-identity proves the relocation is *faithful* — the new
numpy reconciler reproduces the old served numbers exactly. It says **nothing** about
whether per-draw proportional top-down is the *right* method. That question — and the
literature — is the next section.

---
## 🔬 Is it a *good* method? (panel additions)

### A · Where proportional sits in the literature (register C-60)

Hierarchical-forecast reconciliation has a well-developed theory. **Top-down proportional**
(what ships here, FPP3 terminology) is the **pragmatic baseline**: simple, conservative,
and — applied per draw — distribution-aware. But it is known to be *suboptimal*:

* **MinT** (Wickramasuriya et al. 2019) reconciles by minimising the trace of the
  reconciled error covariance — provably the minimum-variance unbiased linear reconciliation.
* **Probabilistic reconciliation** (Panagiotelis et al. 2023) reconciles the *joint
  distribution* coherently, rather than rescaling per draw.

Adopting a principled reconciler is **deliberately deferred** (views-postprocessing C-37):
this package ships the faithful port first, behind a stable interface, so a better method
can be added as a sibling without breaking consumers. **Bit-identity to the torch oracle is
not evidence the method is good — only that the port is faithful.**

### B · Does reconciliation *help*? (against the known truth)

Because the synthetic DGP has a known true cell mean and country total, we can ask whether
reconciling moves estimates *toward* the truth — or just enforces the constraint. The honest
answer is the latter: reconciliation buys **coherence**, and accuracy at the country level
becomes the **country model's** accuracy; per cell it can move *away* from truth when the
country forecast disagrees with the grid.

In [ ]:
# country-level: raw grid sum vs reconciled (== country forecast) vs the TRUE total
true_tot = np.array([sc.country_total_truth[(int(t), int(u))]
                     for t, u in zip(sc.cm_index.time, sc.cm_index.unit)])
raw_country = np.array([                       # raw grid summed to each (time, country)
    sc.pgm_prediction.values[(sc.pgm_index.time == t) & (cm_units == u)].sum(0).mean()
    for t, u in zip(sc.cm_index.time, sc.cm_index.unit)])
recon_country = np.array([
    recon_pgm.values[(sc.pgm_index.time == t) & (cm_units == u)].sum(0).mean()
    for t, u in zip(sc.cm_index.time, sc.cm_index.unit)])
print("country-total mean abs error vs truth:")
print(f"  raw grid sum        : {np.abs(raw_country - true_tot).mean():7.3f}")
print(f"  reconciled (=cm fc) : {np.abs(recon_country - true_tot).mean():7.3f}")

# per-cell: did reconciliation move cells toward or away from their true means?
true_cell = np.array([sc.cell_total_truth[(int(t), int(u))]
                      for t, u in zip(sc.pgm_index.time, sc.pgm_index.unit)])
raw_cell = sc.pgm_prediction.values.mean(1)
rec_cell = recon_pgm.values.mean(1)
print("\nper-cell mean abs error vs truth:")
print(f"  raw grid    : {np.abs(raw_cell - true_cell).mean():7.3f}")
print(f"  reconciled  : {np.abs(rec_cell - true_cell).mean():7.3f}")
print("\nVerdict: reconciliation guarantees coherence (the grid sums to the country, "
      "exactly),\nbut coherence is not accuracy. Here it *improved* the country total "
      "(it adopted the\ntighter country model) yet *worsened* the per-cell estimates "
      "(top-down scaling injects\nthe country forecast's per-draw noise into every cell "
      "and overrides the grid's own\nper-cell signal). Which way it nets depends on the "
      "level you care about and on whether\nthe country model beats the grid aggregate — "
      "exactly the trade a principled reconciler\n(MinT / probabilistic, §A) is designed "
      "to optimise. Coherence is the deliverable here;\naccuracy stays the model's job.")

### C · A spatial / map view (register C-61)

Reconciliation *redistributes* mass across the grid. On the toy lattice (synthetic
coordinates — no real geography), raw vs reconciled point estimates and the change map show
*where* it moves forecasts.

In [ ]:
nr, nc = sc.lattice_shape
m0 = (sc.pgm_index.time == 500)
def to_grid(vals_full):
    g = np.full((nr, nc), np.nan)
    for row in np.where(m0)[0]:
        r, c = sc.coords[int(sc.pgm_index.unit[row])]; g[r, c] = vals_full[row]
    return g

raw_pt = sc.pgm_prediction.values.mean(1)
rec_pt = recon_pgm.values.mean(1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
g_raw, g_rec = to_grid(raw_pt), to_grid(rec_pt)
vmax = np.nanmax([g_raw, g_rec])
for ax, (lab, g, kw) in zip(axes, [
    ("raw grid (mean)", g_raw, dict(cmap="magma", vmin=0, vmax=vmax)),
    ("reconciled (mean)", g_rec, dict(cmap="magma", vmin=0, vmax=vmax)),
    ("change (rec − raw)", g_rec - g_raw, dict(cmap="coolwarm")),
]):
    im = ax.imshow(g, **kw); ax.set_title(lab, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("toy lattice, month 500 — reconciliation redistributes mass (illustrative)", fontsize=10)
plt.tight_layout(); plt.show()

### D · Decision relevance

The reconciled country posterior is what a national food-security decision keys on. A
threshold exceedance on the **coherent** country total — `P(total > c)` — is the readout,
now guaranteed consistent with the grid map above.

In [ ]:
recon_country_dist = aggregate_distributions(recon_pgm, sc.mapping, SpatialLevel.CM)
c70 = int(np.where((recon_country_dist.index.time == 500) & (recon_country_dist.index.unit == 70))[0][0])
draws = recon_country_dist.values[c70]
severe = float(np.quantile(draws, 0.75))
band = hdi(recon_country_dist, mass=0.9)[c70]
print(f"country 70, month 500 — reconciled country total:")
print(f"  90% HDI                 : [{band[0]:.1f}, {band[1]:.1f}]")
print(f"  P(total > {severe:.0f}) (severe) : {float((draws > severe).mean()):.2f}")
print("  this exceedance is coherent with the per-cell grid map by construction.")

## Reconciliation in one screen

| You saw | The takeaway |
|---|---|
| `reconcile_proportional` (§2) | per-draw top-down; conserves, preserves zeros, handles the all-zero edge |
| `ReconciliationModule` (§4) | injected mapping (ADR-014), de-mutated, fail-loud |
| `assert_reconcile_contract` (§5) | published contract; the mapping is load-bearing |
| `ES(aggregate) ≠ Σ ES(cell)` (§6) | joint sampling; tail risk is subadditive |
| bit-identity to the oracle (§7) | proves *faithful relocation*, **not** method quality |
| proportional vs MinT/probabilistic (§A) | a pragmatic baseline; the principled upgrade is deferred (C-37) |
| does-it-help (§B) | reconciliation buys **coherence**, not accuracy — that stays the model's job |

That closes the tour: [`01`](01_frames.ipynb) the storage contract, [`02`](02_summaries.ipynb)
reading honest summaries off a posterior, and `03` making those posteriors coherent across
spatial levels — all on the frozen `views_frames` surface, all on synthetic data.